# Example on how to import and export patient data in pyRadPlan.

pyRadPlan ships a small, extensible import/export framework in `pyRadPlan.io`.
It supports MATLAB (`.mat`, matRad-compatible), DICOM (CT, RTSTRUCT, SEG, RTDOSE),
the SimpleITK-based image formats (NIfTI/NRRD/MetaImage), NumPy `.npz` and pickle.

There are two layers:

1. A simple **top-level API**: `load_patient`, `load_data` and `save_data`.
2. **Low-level handlers** (`MatlabHandler`, `DicomHandler`) for fine-grained control.

To display this script in a Jupyter Notebook, install jupytext via pip and run:

```bash
pip install jupytext
jupytext --to notebook path/to/this/file/utils_io.py
```

In [1]:
# some imports
import tempfile
from pathlib import Path

from pyRadPlan import (
    load_tg119,
    load_patient,
    load_data,
    save_data,
)
from pyRadPlan.io import MatlabHandler, DicomHandler

C:\NiklasLocal\python\pyRadPlan\.venv313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Top-level API

The most convenient entry point. `load_patient` returns the CT image (CT) and the
Structure Set (CST), choosing the right loader automatically from the path
(a `.mat` file, or a folder/file of DICOM data).

In [2]:
# Load the TG119 phantom bundled with pyRadPlan ...
ct, cst = load_tg119()

# ... which is equivalent to loading the patient file directly:
from importlib import resources  # noqa: E402

tg119_path = resources.files("pyRadPlan.data.phantoms").joinpath("TG119.mat")
ct, cst = load_patient(tg119_path)

`load_data` loads *everything* it finds into a dictionary. For a matRad `.mat` file this
may contain `ct`, `cst` and (if present) `dose`; for a DICOM folder it collects the CT
series, structures and dose. Missing pieces are simply omitted.

In [3]:
data = load_data(tg119_path)
print("Loaded keys:", list(data))
ct = data["ct"]
cst = data["cst"]

Loaded keys: ['ct', 'cst']


## Saving data

`save_data` writes one or more objects. The format is chosen (in this order) from an
explicit `format=` argument, the extension of `file_name`, or a fast default (`.mat`).

In [4]:
work_dir = Path(tempfile.mkdtemp())

# Save into a single .mat file (format taken from the extension):
save_data(ct=ct, cst=cst, file_name=str(work_dir / "patient.mat"))

# No extension? The default format is appended automatically (-> patient2.mat):
save_data(ct=ct, file_name=str(work_dir / "patient2"))

# Force a format explicitly, regardless of the file name:
save_data(ct=ct, file_name=str(work_dir / "patient3"), format="mat")

# A dict of named objects works too:
save_data({"ct": ct, "cst": cst}, file_name=str(work_dir / "patient4.mat"))

print("Wrote:", sorted(p.name for p in work_dir.glob("*.mat")))

Wrote: ['patient.mat', 'patient2.mat', 'patient3.mat', 'patient4.mat']


## Low-level handlers

Each format also has a handler that bundles importing and exporting and lets you load
individual objects. A `MatlabHandler` is bound to a single file.

In [5]:
handler = MatlabHandler(work_dir / "patient.mat")
ct = handler.load_ct()  # load just the CT
cst = handler.load_cst(ct)  # load just the StructureSet
# handler.load_patient() -> (ct, cst); handler.load_data() -> dict of everything

# Saving via the handler is equivalent to save_data with that format:
handler_out = MatlabHandler(work_dir / "from_handler.mat")
handler_out.save(ct=ct, cst=cst)

## DICOM import / export

DICOM is directory-based. Here we export the phantom to a folder as a CT series plus an
RTSTRUCT, then import it back. CT geometry/HU and structure masks are preserved.

In [6]:
dicom_dir = work_dir / "dicom"
DicomHandler(dicom_dir).save(ct=ct, cst=cst)
print("DICOM files:", sorted(p.name for p in dicom_dir.glob("*.dcm")))

# Re-import the whole folder:
ct_dcm, cst_dcm = load_patient(dicom_dir)
print("Imported structures:", [voi.name for voi in cst_dcm.vois])

DICOM files: ['CT_0001.dcm', 'CT_0002.dcm', 'CT_0003.dcm', 'CT_0004.dcm', 'CT_0005.dcm', 'CT_0006.dcm', 'CT_0007.dcm', 'CT_0008.dcm', 'CT_0009.dcm', 'CT_0010.dcm', 'CT_0011.dcm', 'CT_0012.dcm', 'CT_0013.dcm', 'CT_0014.dcm', 'CT_0015.dcm', 'CT_0016.dcm', 'CT_0017.dcm', 'CT_0018.dcm', 'CT_0019.dcm', 'CT_0020.dcm', 'CT_0021.dcm', 'CT_0022.dcm', 'CT_0023.dcm', 'CT_0024.dcm', 'CT_0025.dcm', 'CT_0026.dcm', 'CT_0027.dcm', 'CT_0028.dcm', 'CT_0029.dcm', 'CT_0030.dcm', 'CT_0031.dcm', 'CT_0032.dcm', 'CT_0033.dcm', 'CT_0034.dcm', 'CT_0035.dcm', 'CT_0036.dcm', 'CT_0037.dcm', 'CT_0038.dcm', 'CT_0039.dcm', 'CT_0040.dcm', 'CT_0041.dcm', 'CT_0042.dcm', 'CT_0043.dcm', 'CT_0044.dcm', 'CT_0045.dcm', 'CT_0046.dcm', 'CT_0047.dcm', 'CT_0048.dcm', 'CT_0049.dcm', 'CT_0050.dcm', 'CT_0051.dcm', 'CT_0052.dcm', 'CT_0053.dcm', 'CT_0054.dcm', 'CT_0055.dcm', 'CT_0056.dcm', 'CT_0057.dcm', 'CT_0058.dcm', 'CT_0059.dcm', 'CT_0060.dcm', 'CT_0061.dcm', 'CT_0062.dcm', 'CT_0063.dcm', 'CT_0064.dcm', 'CT_0065.dcm', 'CT_0066.dc

Imported structures: ['Core', 'OuterTarget', 'BODY']


Structures are exported as RTSTRUCT by default. To export them as a DICOM SEG object
instead (which stores the voxel masks directly), use the exporter's `structure_format`:

In [7]:
from pyRadPlan.io.dicom import DicomExporter  # noqa: E402

seg_dir = work_dir / "dicom_seg"
DicomExporter(seg_dir, structure_format="seg").save(ct=ct, cst=cst)
print("SEG export:", sorted(p.name for p in seg_dir.glob("*.dcm")))

SEG export:

 ['CT_0001.dcm', 'CT_0002.dcm', 'CT_0003.dcm', 'CT_0004.dcm', 'CT_0005.dcm', 'CT_0006.dcm', 'CT_0007.dcm', 'CT_0008.dcm', 'CT_0009.dcm', 'CT_0010.dcm', 'CT_0011.dcm', 'CT_0012.dcm', 'CT_0013.dcm', 'CT_0014.dcm', 'CT_0015.dcm', 'CT_0016.dcm', 'CT_0017.dcm', 'CT_0018.dcm', 'CT_0019.dcm', 'CT_0020.dcm', 'CT_0021.dcm', 'CT_0022.dcm', 'CT_0023.dcm', 'CT_0024.dcm', 'CT_0025.dcm', 'CT_0026.dcm', 'CT_0027.dcm', 'CT_0028.dcm', 'CT_0029.dcm', 'CT_0030.dcm', 'CT_0031.dcm', 'CT_0032.dcm', 'CT_0033.dcm', 'CT_0034.dcm', 'CT_0035.dcm', 'CT_0036.dcm', 'CT_0037.dcm', 'CT_0038.dcm', 'CT_0039.dcm', 'CT_0040.dcm', 'CT_0041.dcm', 'CT_0042.dcm', 'CT_0043.dcm', 'CT_0044.dcm', 'CT_0045.dcm', 'CT_0046.dcm', 'CT_0047.dcm', 'CT_0048.dcm', 'CT_0049.dcm', 'CT_0050.dcm', 'CT_0051.dcm', 'CT_0052.dcm', 'CT_0053.dcm', 'CT_0054.dcm', 'CT_0055.dcm', 'CT_0056.dcm', 'CT_0057.dcm', 'CT_0058.dcm', 'CT_0059.dcm', 'CT_0060.dcm', 'CT_0061.dcm', 'CT_0062.dcm', 'CT_0063.dcm', 'CT_0064.dcm', 'CT_0065.dcm', 'CT_0066.dcm', 'CT_0067

That's it! The framework also accepts a dose distribution (a `SimpleITK.Image`) via the
`dose=` argument of `save_data` / the handlers, which is written as DICOM RTDOSE or stored
in the matRad `resultGUI`. See `utils_matrad.py` for matRad-specific (de)serialization.